In [11]:
import csv
import time
import os
import urllib.request
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service

def download_skyview_fits(csv_file, download_folder):
  
    # Setup Chrome options - keep browser stable
    chrome_options = Options()
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    chrome_options.add_argument('--disable-gpu')
    chrome_options.add_argument('--window-size=1920,1080')
    
    prefs = {
        "download.default_directory": os.path.abspath(download_folder),
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "safebrowsing.enabled": True
    }
    chrome_options.add_experimental_option("prefs", prefs)
    
    # Initialize the browser with ChromeDriver
    chromedriver_path = r'C:\Users\aarad\.wdm\drivers\chromedriver\win64\141.0.7390.122\chromedriver-win32\chromedriver.exe'
    service = Service(chromedriver_path)
    driver = webdriver.Chrome(service=service, options=chrome_options)
    wait = WebDriverWait(driver, 30)
    
    try:
        # Read CSV file and skip header rows
        with open(csv_file, 'r') as f:
            reader = csv.reader(f)
            all_rows = list(reader)
            
        # Skip first 2 rows (headers) and get coordinates from first column
        coordinates = []
        for row in all_rows[2:]:  # Start from row 3 (index 2)
            if row and row[0].strip():
                coordinates.append(row[0].strip())
        
        print(f"Found {len(coordinates)} coordinates to process")
        print(f"Skipped first 2 header rows\n")
        
        successful_downloads = 0
        failed_downloads = 0
        
        # Process each coordinate
        for idx, coord in enumerate(coordinates, 1):
            print(f"[{idx}/{len(coordinates)}] Processing: {coord}")
            
            try:
                # Navigate to SkyView
                driver.get("https://skyview.gsfc.nasa.gov/current/cgi/query.pl")
                time.sleep(2)
                
                # Enter coordinates in the search box
                search_box = wait.until(
                    EC.presence_of_element_located((By.NAME, "Position"))
                )
                search_box.clear()
                search_box.send_keys(coord)
                print(f"  → Entered coordinates")
                
                # Make sure "Display results in new window" is checked
                try:
                    new_window_checkbox = driver.find_element(By.XPATH, "//input[@type='checkbox' and contains(@onclick, 'window')]")
                    if not new_window_checkbox.is_selected():
                        new_window_checkbox.click()
                        print(f"  → Enabled 'Display results in new window'")
                except:
                    print(f"  → 'Display results in new window' already enabled or not found")
                
                # Find the UV dropdown/select element and choose GALEX Near UV
                try:
                    # Look for the select dropdown in the UV section
                    uv_select = driver.find_element(By.XPATH, "//select[.//option[contains(text(), 'GALEX Near UV')]]")
                    
                    # Scroll to it
                    driver.execute_script("arguments[0].scrollIntoView(true);", uv_select)
                    time.sleep(0.5)
                    
                    # Select GALEX Near UV from dropdown
                    from selenium.webdriver.support.ui import Select
                    select = Select(uv_select)
                    select.select_by_visible_text("GALEX Near UV")
                    print(f"  → Selected GALEX Near UV from dropdown")
                    
                except Exception as e:
                    print(f"  → Trying alternate method to select GALEX Near UV...")
                    # Alternative: click the option directly
                    galex_option = driver.find_element(By.XPATH, "//option[contains(text(), 'GALEX Near UV')]")
                    driver.execute_script("arguments[0].scrollIntoView(true);", galex_option)
                    time.sleep(0.3)
                    galex_option.click()
                    print(f"  → Selected GALEX Near UV")
                
                # Click the Submit Request button (near "Initiate request:")
                try:
                    submit_button = wait.until(
                        EC.element_to_be_clickable((By.XPATH, "//input[@type='button' and @value='Submit Request']"))
                    )
                except:
                    # Try alternate locator
                    submit_button = driver.find_element(By.XPATH, "//input[@value='Submit Request']")
                
                driver.execute_script("arguments[0].scrollIntoView(true);", submit_button)
                time.sleep(0.5)
                
                # Store current window handle
                main_window = driver.current_window_handle
                
                submit_button.click()
                print(f"  → Clicked Submit button, waiting for new window...")
                
                # Wait for new window to open
                time.sleep(3)
                
                # Switch to the new window
                all_windows = driver.window_handles
                for window in all_windows:
                    if window != main_window:
                        driver.switch_to.window(window)
                        print(f"  → Switched to results window")
                        break
                
                # Wait for results page to load
                print(f"  → Waiting for FITS file generation...")
                time.sleep(10)
                
                # Try to find and download FITS file
                fits_downloaded = False
                fits_url = None
                
                # Look for FITS download link
                try:
                    # Find all links on the page
                    all_links = driver.find_elements(By.TAG_NAME, "a")
                    
                    for link in all_links:
                        href = link.get_attribute('href')
                        if href and '.fits' in href.lower():
                            fits_url = href
                            print(f"  → Found FITS URL: {fits_url}")
                            break
                    
                    if fits_url:
                        # Download using the coordinate name
                        safe_filename = coord.replace(' ', '_').replace('/', '-')
                        output_path = os.path.join(download_folder, f"{safe_filename}.fits")
                        
                        # Download the file directly using urllib
                        print(f"  → Downloading to: {safe_filename}.fits")
                        urllib.request.urlretrieve(fits_url, output_path)
                        
                        print(f"  ✓ Successfully downloaded!\n")
                        successful_downloads += 1
                        fits_downloaded = True
                    
                except Exception as e:
                    print(f"  ✗ Error finding/downloading FITS: {e}")
                
                if not fits_downloaded:
                    print(f"  ✗ Could not download FITS for {coord}\n")
                    failed_downloads += 1
                    
                    # Save debug page
                    debug_file = os.path.join(download_folder, f"debug_{safe_filename}.html")
                    with open(debug_file, 'w', encoding='utf-8') as f:
                        f.write(driver.page_source)
                    print(f"  (Debug page saved: debug_{safe_filename}.html)\n")
                
                # Close the results window and switch back to main window
                driver.close()
                driver.switch_to.window(main_window)
                print(f"  → Closed results window, back to main form")
                
            except Exception as e:
                print(f"  ✗ Error processing {coord}: {str(e)[:100]}\n")
                failed_downloads += 1
                
                # Try to close any extra windows and go back to main
                try:
                    all_windows = driver.window_handles
                    if len(all_windows) > 1:
                        driver.close()
                        driver.switch_to.window(all_windows[0])
                except:
                    pass
                continue
            
            # Small delay between requests to be respectful
            time.sleep(2)
        
        print("\n" + "="*60)
        print(f"SUMMARY:")
        print(f"  Total processed: {len(coordinates)}")
        print(f"  ✓ Successful: {successful_downloads}")
        print(f"  ✗ Failed: {failed_downloads}")
        print(f"  Files saved to: {os.path.abspath(download_folder)}")
        print("="*60)
        
    except Exception as e:
        print(f"\n✗ Fatal error: {e}")
        import traceback
        traceback.print_exc()
    finally:
        print("\nClosing browser...")
        driver.quit()

if __name__ == "__main__":
    # Configuration
    CSV_FILE = "coordinates.csv"  # Change to your CSV file path
    DOWNLOAD_FOLDER = "fits_downloads"  # Change to your folder path
    
    # Create download folder if it doesn't exist
    if not os.path.exists(DOWNLOAD_FOLDER):
        os.makedirs(DOWNLOAD_FOLDER)
        print(f"Created folder: {DOWNLOAD_FOLDER}\n")
    
    download_skyview_fits(CSV_FILE, DOWNLOAD_FOLDER)
    print("\nScript completed!")

Found 606 coordinates to process
Skipped first 2 header rows

[1/606] Processing: NGC 0024
  → Entered coordinates
  → 'Display results in new window' already enabled or not found
  → Selected GALEX Near UV from dropdown
  → Clicked Submit button, waiting for new window...
  → Switched to results window
  → Waiting for FITS file generation...
  → Found FITS URL: https://skyview.gsfc.nasa.gov/tempspace/fits/skv1053948116397.fits
  → Downloading to: NGC_0024.fits
  ✓ Successfully downloaded!

  → Closed results window, back to main form
[2/606] Processing: NGC 0055
  → Entered coordinates
  → 'Display results in new window' already enabled or not found
  → Selected GALEX Near UV from dropdown
  → Clicked Submit button, waiting for new window...
  → Switched to results window
  → Waiting for FITS file generation...
  → Found FITS URL: https://skyview.gsfc.nasa.gov/tempspace/fits/skv1054001239744.fits
  → Downloading to: NGC_0055.fits
  ✓ Successfully downloaded!

  → Closed results window

In [1]:
import csv
import time
import os
import urllib.request
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service

def download_skyview_fits(csv_file, download_folder):
    """
    Automate SkyView GALEX Near UV FITS downloads from CSV coordinates
    
    Args:
        csv_file: Path to CSV file with coordinates in first column
        download_folder: Path to folder where FITS files will be saved
    """
    
    # Setup Chrome options - keep browser stable
    chrome_options = Options()
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    chrome_options.add_argument('--disable-gpu')
    chrome_options.add_argument('--window-size=1920,1080')
    
    prefs = {
        "download.default_directory": os.path.abspath(download_folder),
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "safebrowsing.enabled": True
    }
    chrome_options.add_experimental_option("prefs", prefs)
    
    # Initialize the browser with ChromeDriver
    chromedriver_path = r'C:\Users\aarad\.wdm\drivers\chromedriver\win64\141.0.7390.122\chromedriver-win32\chromedriver.exe'
    service = Service(chromedriver_path)
    driver = webdriver.Chrome(service=service, options=chrome_options)
    wait = WebDriverWait(driver, 30)
    
    try:
        # Read CSV file and skip header rows
        with open(csv_file, 'r') as f:
            reader = csv.reader(f)
            all_rows = list(reader)
            
        # Skip first 2 rows (headers) and get coordinates from first column
        # START_ROW: Change this to start from a specific row (e.g., 188)
        START_ROW = 188  # Row number to start from (1-indexed, includes header rows)
        
        coordinates = []
        for i, row in enumerate(all_rows[2:], start=3):  # Start counting from row 3
            if row and row[0].strip():
                if i >= START_ROW:  # Only process from START_ROW onwards
                    coordinates.append(row[0].strip())
        
        print(f"Starting from row {START_ROW}")
        print(f"Found {len(coordinates)} coordinates to process")
        print(f"Skipped first {START_ROW - 1} rows\n")
        
        successful_downloads = 0
        failed_downloads = 0
        
        # Process each coordinate
        for idx, coord in enumerate(coordinates, 1):
            print(f"[{idx}/{len(coordinates)}] Processing: {coord}")
            
            try:
                # Navigate to SkyView
                driver.get("https://skyview.gsfc.nasa.gov/current/cgi/query.pl")
                time.sleep(2)
                
                # Enter coordinates in the search box
                search_box = wait.until(
                    EC.presence_of_element_located((By.NAME, "Position"))
                )
                search_box.clear()
                search_box.send_keys(coord)
                print(f"  → Entered coordinates")
                
                # Make sure "Display results in new window" is checked
                try:
                    new_window_checkbox = driver.find_element(By.XPATH, "//input[@type='checkbox' and contains(@onclick, 'window')]")
                    if not new_window_checkbox.is_selected():
                        new_window_checkbox.click()
                        print(f"  → Enabled 'Display results in new window'")
                except:
                    print(f"  → 'Display results in new window' already enabled or not found")
                
                # Find the UV dropdown/select element and choose GALEX Near UV
                try:
                    # Look for the select dropdown in the UV section
                    uv_select = driver.find_element(By.XPATH, "//select[.//option[contains(text(), 'GALEX Near UV')]]")
                    
                    # Scroll to it
                    driver.execute_script("arguments[0].scrollIntoView(true);", uv_select)
                    time.sleep(0.5)
                    
                    # Select GALEX Near UV from dropdown
                    from selenium.webdriver.support.ui import Select
                    select = Select(uv_select)
                    select.select_by_visible_text("GALEX Near UV")
                    print(f"  → Selected GALEX Near UV from dropdown")
                    
                except Exception as e:
                    print(f"  → Trying alternate method to select GALEX Near UV...")
                    # Alternative: click the option directly
                    galex_option = driver.find_element(By.XPATH, "//option[contains(text(), 'GALEX Near UV')]")
                    driver.execute_script("arguments[0].scrollIntoView(true);", galex_option)
                    time.sleep(0.3)
                    galex_option.click()
                    print(f"  → Selected GALEX Near UV")
                
                # Click the Submit Request button (near "Initiate request:")
                try:
                    submit_button = wait.until(
                        EC.element_to_be_clickable((By.XPATH, "//input[@type='button' and @value='Submit Request']"))
                    )
                except:
                    # Try alternate locator
                    submit_button = driver.find_element(By.XPATH, "//input[@value='Submit Request']")
                
                driver.execute_script("arguments[0].scrollIntoView(true);", submit_button)
                time.sleep(0.5)
                
                # Store current window handle
                main_window = driver.current_window_handle
                
                submit_button.click()
                print(f"  → Clicked Submit button, waiting for new window...")
                
                # Wait for new window to open
                time.sleep(3)
                
                # Switch to the new window
                all_windows = driver.window_handles
                for window in all_windows:
                    if window != main_window:
                        driver.switch_to.window(window)
                        print(f"  → Switched to results window")
                        break
                
                # Wait for results page to load
                print(f"  → Waiting for FITS file generation...")
                time.sleep(10)
                
                # Try to find and download FITS file
                fits_downloaded = False
                fits_url = None
                
                # Look for FITS download link
                try:
                    # Find all links on the page
                    all_links = driver.find_elements(By.TAG_NAME, "a")
                    
                    for link in all_links:
                        href = link.get_attribute('href')
                        if href and '.fits' in href.lower():
                            fits_url = href
                            print(f"  → Found FITS URL: {fits_url}")
                            break
                    
                    if fits_url:
                        # Download using the coordinate name
                        safe_filename = coord.replace(' ', '_').replace('/', '-')
                        output_path = os.path.join(download_folder, f"{safe_filename}.fits")
                        
                        # Download the file directly using urllib
                        print(f"  → Downloading to: {safe_filename}.fits")
                        urllib.request.urlretrieve(fits_url, output_path)
                        
                        print(f"  ✓ Successfully downloaded!\n")
                        successful_downloads += 1
                        fits_downloaded = True
                    
                except Exception as e:
                    print(f"  ✗ Error finding/downloading FITS: {e}")
                
                if not fits_downloaded:
                    print(f"  ✗ Could not download FITS for {coord}\n")
                    failed_downloads += 1
                    
                    # Save debug page
                    debug_file = os.path.join(download_folder, f"debug_{safe_filename}.html")
                    with open(debug_file, 'w', encoding='utf-8') as f:
                        f.write(driver.page_source)
                    print(f"  (Debug page saved: debug_{safe_filename}.html)\n")
                
                # Close the results window and switch back to main window
                driver.close()
                driver.switch_to.window(main_window)
                print(f"  → Closed results window, back to main form")
                
            except Exception as e:
                print(f"  ✗ Error processing {coord}: {str(e)[:100]}\n")
                failed_downloads += 1
                
                # Try to close any extra windows and go back to main
                try:
                    all_windows = driver.window_handles
                    if len(all_windows) > 1:
                        driver.close()
                        driver.switch_to.window(all_windows[0])
                except:
                    pass
                continue
            
            # Small delay between requests to be respectful
            time.sleep(2)
        
        print("\n" + "="*60)
        print(f"SUMMARY:")
        print(f"  Total processed: {len(coordinates)}")
        print(f"  ✓ Successful: {successful_downloads}")
        print(f"  ✗ Failed: {failed_downloads}")
        print(f"  Files saved to: {os.path.abspath(download_folder)}")
        print("="*60)
        
    except Exception as e:
        print(f"\n✗ Fatal error: {e}")
        import traceback
        traceback.print_exc()
    finally:
        print("\nClosing browser...")
        driver.quit()

if __name__ == "__main__":
    # Configuration
    CSV_FILE = "coordinates.csv"  # Change to your CSV file path
    DOWNLOAD_FOLDER = "fits_downloads"  # Change to your folder path
    
    # Create download folder if it doesn't exist
    if not os.path.exists(DOWNLOAD_FOLDER):
        os.makedirs(DOWNLOAD_FOLDER)
        print(f"Created folder: {DOWNLOAD_FOLDER}\n")
    
    download_skyview_fits(CSV_FILE, DOWNLOAD_FOLDER)
    print("\nScript completed!")

Starting from row 188
Found 421 coordinates to process
Skipped first 187 rows

[1/421] Processing: ESO 034-G013
  → Entered coordinates
  → 'Display results in new window' already enabled or not found
  → Selected GALEX Near UV from dropdown
  → Clicked Submit button, waiting for new window...
  → Switched to results window
  → Waiting for FITS file generation...
  → Found FITS URL: https://skyview.gsfc.nasa.gov/tempspace/fits/skv1122166494266.fits
  → Downloading to: ESO_034-G013.fits
  ✓ Successfully downloaded!

  → Closed results window, back to main form
[2/421] Processing: NGC 2442
  → Entered coordinates
  → 'Display results in new window' already enabled or not found
  → Selected GALEX Near UV from dropdown
  → Clicked Submit button, waiting for new window...
  → Switched to results window
  → Waiting for FITS file generation...
  → Found FITS URL: https://skyview.gsfc.nasa.gov/tempspace/fits/skv1122221121899.fits
  → Downloading to: NGC_2442.fits
  ✓ Successfully downloaded!

